# 09. Portfolio Construction / Execution Layer

Build survivor-count agnostic portfolio candidates from the Notebook 8 core survivor alpha library.

Scope guardrails: this notebook consumes only `PROMOTE_CORE` rows from `survivor_alpha_registry_current` and matching values from `pre_ml_alpha_inputs_current`. It does not use regime overlay diagnostics, REVIEW_SATELLITE alphas, rejected alphas, ML, or any hard-coded alpha names.

## 1. Purpose and scope

Use the core production path outputs from Notebook 08 to create dynamic survivor alpha pools, construct long-only and long-short portfolios, estimate execution-lagged transaction-cost-aware returns, compare against SPY when available, and write auditable portfolio artifacts.

## 2. Imports and config

In [9]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "3-Phase 3_Portfolio Construction":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import load_benchmark_prices, load_ohlcv_panels, load_table, table_exists
from src.portfolio_backtest import compute_portfolio_metrics, compute_strategy_returns
from src.portfolio_construction import (
    apply_rebalance_schedule,
    build_alpha_signal_stack,
    build_long_only_top_bucket_positions,
    build_survivor_weight_table,
    build_target_positions,
    cap_position_weights,
    combine_survivor_alphas,
    compute_turnover,
    filter_pre_ml_alpha_inputs_to_survivors,
    load_pre_ml_alpha_inputs,
    load_survivor_alpha_registry,
    normalize_cross_sectional_scores,
    renormalize_long_short,
    select_promote_core_survivors,
)
from src.portfolio_storage import DYNAMIC_PORTFOLIO_TABLES, save_dynamic_portfolio_outputs
from src.run_config import get_sqlite_db_path, make_run_id, make_run_timestamp

DB_PATH = get_sqlite_db_path()
PORTFOLIO_VERSION = "phase9_survivor_portfolio_v1"
MAX_ABS_WEIGHT = 0.05
TOP_QUANTILE = 0.20
BOTTOM_QUANTILE = 0.20
GROSS_EXPOSURE = 1.0
REBALANCE_FREQUENCY = 5
COST_BPS = 5
EXECUTION_LAG = 1
PORTFOLIO_MODES = ["long_only_top", "long_short_top_bottom"]


## 3. Create portfolio run_id / timestamp

In [10]:
run_id = make_run_id(prefix="phase9_nb09_portfolio")
run_timestamp = make_run_timestamp()

run_id, run_timestamp


('phase9_nb09_portfolio_20260511_095939', '2026-05-11 09:59:39')

## 4. Load PROMOTE_CORE survivors from Notebook 8

In [11]:
survivor_registry_all = load_survivor_alpha_registry()
survivor_registry = select_promote_core_survivors(survivor_registry_all)
pre_ml_alpha_inputs_all = load_pre_ml_alpha_inputs()
pre_ml_alpha_inputs = filter_pre_ml_alpha_inputs_to_survivors(
    pre_ml_alpha_inputs_all,
    survivor_registry,
)

survivor_names = sorted(survivor_registry["alpha_name"].dropna().unique()) if "alpha_name" in survivor_registry.columns else []
input_alpha_names = sorted(pre_ml_alpha_inputs["alpha_name"].dropna().unique()) if "alpha_name" in pre_ml_alpha_inputs.columns else []
decision_col = ("promotion_decision_final"
    if "promotion_decision_final" in survivor_registry_all.columns
    else "promotion_decision")

excluded_non_core = (
    survivor_registry_all.loc[
        ~survivor_registry_all[decision_col].eq("PROMOTE_CORE"),
        ["alpha_name", decision_col],
    ].drop_duplicates()
    if not survivor_registry_all.empty and {"alpha_name", decision_col}.issubset(survivor_registry_all.columns)
    else pd.DataFrame(columns=["alpha_name", decision_col])
)

excluded_non_core = excluded_non_core.rename(columns={decision_col: "promotion_decision"})

overlay_name_overlap = []
if table_exists("regime_context_alpha_metadata_current", db_path=DB_PATH):
    overlay_metadata = load_table("regime_context_alpha_metadata_current", db_path=DB_PATH)
    overlay_name_overlap = sorted(set(survivor_names).intersection(set(overlay_metadata["alpha_name"].dropna())))

no_core_survivors = survivor_registry.empty
if no_core_survivors:
    print("No PROMOTE_CORE survivors found in survivor_alpha_registry_current; Notebook 09 will write empty portfolio outputs for lineage consistency.")
if set(input_alpha_names) != set(survivor_names):
    missing_inputs = sorted(set(survivor_names).difference(input_alpha_names))
    extra_inputs = sorted(set(input_alpha_names).difference(survivor_names))
    raise ValueError(f"Pre-ML alpha input mismatch. Missing={missing_inputs}, extra={extra_inputs}")
if overlay_name_overlap:
    raise ValueError(f"Regime overlay names leaked into survivor selection: {overlay_name_overlap}")

decision_col = (
    "promotion_decision_final"
    if "promotion_decision_final" in survivor_registry_all.columns
    else "promotion_decision")
source_counts = (
    survivor_registry_all[decision_col].value_counts(dropna=False).rename_axis("promotion_decision").reset_index(name="n_rows")
    if not survivor_registry_all.empty and decision_col in survivor_registry_all.columns
    else pd.DataFrame(columns=["promotion_decision", "n_rows"])
)

print(f"PROMOTE_CORE survivor count: {len(survivor_names)}")
print(survivor_names)
print(f"Filtered pre-ML rows: {len(pre_ml_alpha_inputs):,}")

display(source_counts)
display(survivor_registry)
display(excluded_non_core)


PROMOTE_CORE survivor count: 1
['alpha_regime_blend_dynamic_v4_smooth']
Filtered pre-ML rows: 1,002,844


,promotion_decision,n_rows
0,PROMOTE_CORE,1
1,REVIEW_SATELLITE,1


,survivor_id,alpha_name,horizon,alpha_sleeve,original_promotion_decision,promotion_decision_final,final_status,alpha_role,survivor_tier,survivor_selection_score,...,stress_status,source_wfv_status,failure_category,interpretation_notes,stress_version,alpha_construction_version,date_frozen,survivor_version,run_id,timestamp_frozen
0,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,CORE_REGIME,REVIEW_SATELLITE,PROMOTE_CORE,CORE_ALPHA_SURVIVOR,CORE_ALPHA,WATCH_STRESS_SURVIVOR,49.596981,...,APPROVED_STRESS,WATCHLIST_CONSTRUCTED_ALPHA_WFV,NONE,High average performance but failed catastroph...,phase7_dynamic_alpha_stress_v3,phase4a_alpha_construction_v4,2026-05-11,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437,2026-05-11 08:44:37


,alpha_name,promotion_decision
1,alpha_hybrid_adaptive_v4_smooth,REVIEW_SATELLITE


## 5. Load clean close prices and benchmark

In [12]:
ohlcv = load_ohlcv_panels(current=True, db_path=DB_PATH)
close_prices = ohlcv["close"].sort_index().sort_index(axis=1)

benchmark_prices = load_benchmark_prices(current=True, db_path=DB_PATH)
if "SPY" in benchmark_prices.columns:
    benchmark_source = "benchmark_prices_current.SPY"
    spy_returns = benchmark_prices["SPY"].pct_change(fill_method=None)
elif "SPY" in close_prices.columns:
    benchmark_source = "clean_close_prices_current.SPY"
    spy_returns = close_prices["SPY"].pct_change(fill_method=None)
else:
    benchmark_source = "SPY_unavailable"
    spy_returns = None

benchmark_summary = pd.DataFrame([
    {
        "close_shape": str(close_prices.shape),
        "benchmark_source": benchmark_source,
        "spy_available": spy_returns is not None,
        "cost_bps": COST_BPS,
        "execution_lag": EXECUTION_LAG,
    }
])

display(benchmark_summary)
display(close_prices.head())

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after loading portfolio OHLCV and benchmark panels')


,close_shape,benchmark_source,spy_available,cost_bps,execution_lag
0,"(2098, 478)",benchmark_prices_current.SPY,True,5,1


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Build survivor alpha panels and portfolio pool weights

In [13]:
alpha_panels = build_alpha_signal_stack(pre_ml_alpha_inputs) if not pre_ml_alpha_inputs.empty else {}
alpha_panels = {name: panel for name, panel in alpha_panels.items() if name in survivor_names}
portfolio_alpha_pool = build_survivor_weight_table(survivor_registry) if not survivor_registry.empty else pd.DataFrame(
    columns=[
        "portfolio_method",
        "alpha_name",
        "horizon",
        "component_weight",
        "pass_rate",
        "worst_degradation",
        "avg_turnover_proxy",
        "promotion_decision_final",
    ]
)

alpha_panel_shapes = pd.DataFrame(
    [
        {"alpha_name": alpha_name, "rows": panel.shape[0], "columns": panel.shape[1]}
        for alpha_name, panel in alpha_panels.items()
    ]
)

weight_sums = (
    portfolio_alpha_pool.groupby("portfolio_method")["component_weight"].sum().reset_index(name="weight_sum")
    if not portfolio_alpha_pool.empty
    else pd.DataFrame(columns=["portfolio_method", "weight_sum"])
)

print(f"Alpha panels built: {len(alpha_panels)}")
display(alpha_panel_shapes)
display(portfolio_alpha_pool)
display(weight_sums)


Alpha panels built: 1


,alpha_name,rows,columns
0,alpha_regime_blend_dynamic_v4_smooth,2053,462


,survivor_id,alpha_name,horizon,survivor_tier,alpha_role,pass_rate,worst_degradation,avg_turnover_proxy,turnover_risk_flag,stress_version,survivor_version,portfolio_method,component_weight,raw_weight_score,weighting_rule
0,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,WATCH_STRESS_SURVIVOR,CORE_ALPHA,0.888889,0.422319,1.746788,LOW_TURNOVER_RISK,phase7_dynamic_alpha_stress_v3,phase8_cluster_aware_survivor_v5,equal_weight_survivors,1.0,1.000000,equal_weight_survivors
1,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,WATCH_STRESS_SURVIVOR,CORE_ALPHA,0.888889,0.422319,1.746788,LOW_TURNOVER_RISK,phase7_dynamic_alpha_stress_v3,phase8_cluster_aware_survivor_v5,stress_score_weighted_survivors,1.0,0.624958,stress_score_weighted_survivors
2,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,WATCH_STRESS_SURVIVOR,CORE_ALPHA,0.888889,0.422319,1.746788,LOW_TURNOVER_RISK,phase7_dynamic_alpha_stress_v3,phase8_cluster_aware_survivor_v5,inverse_turnover_weighted_survivors,1.0,0.572479,inverse_turnover_weighted_survivors
3,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,WATCH_STRESS_SURVIVOR,CORE_ALPHA,0.888889,0.422319,1.746788,LOW_TURNOVER_RISK,phase7_dynamic_alpha_stress_v3,phase8_cluster_aware_survivor_v5,hybrid_survivor_weighted_portfolio,1.0,2.000000,hybrid_survivor_weighted_portfolio


,portfolio_method,weight_sum
0,equal_weight_survivors,1.0
1,hybrid_survivor_weighted_portfolio,1.0
2,inverse_turnover_weighted_survivors,1.0
3,stress_score_weighted_survivors,1.0


## 7. Construct dynamic portfolio candidates

In [14]:
def normalize_long_only(positions: pd.DataFrame, gross_exposure: float = 1.0) -> pd.DataFrame:
    output = positions.astype(float).fillna(0.0).clip(lower=0.0).copy()
    row_gross = output.sum(axis=1)
    active = row_gross.gt(0)
    output.loc[active] = output.loc[active].div(row_gross.loc[active], axis=0).mul(gross_exposure)
    output.loc[~active] = 0.0
    return output.sort_index().sort_index(axis=1)


def panel_to_long(panel: pd.DataFrame, value_name: str, portfolio_method: str, portfolio_mode: str | None = None) -> pd.DataFrame:
    output = panel.stack(dropna=False).rename(value_name).reset_index()
    output = output.rename(columns={"level_0": "Date", "level_1": "ticker"})
    if "Date" not in output.columns:
        output = output.rename(columns={output.columns[0]: "Date"})
    if "ticker" not in output.columns:
        output = output.rename(columns={output.columns[1]: "ticker"})
    output.insert(1, "portfolio_method", portfolio_method)
    if portfolio_mode is not None:
        output.insert(2, "portfolio_mode", portfolio_mode)
    return output

portfolio_score_records = []
portfolio_weight_records = []
portfolio_return_records = []
portfolio_summary_records = []

for portfolio_method in sorted(portfolio_alpha_pool["portfolio_method"].unique()):
    method_weights = portfolio_alpha_pool.loc[
        portfolio_alpha_pool["portfolio_method"].eq(portfolio_method)
    ].set_index("alpha_name")["component_weight"]
    raw_score = combine_survivor_alphas(alpha_panels, method="custom_weight", weights=method_weights)
    normalized_score = normalize_cross_sectional_scores(raw_score)
    portfolio_score_records.append(panel_to_long(normalized_score, "combined_alpha_score", portfolio_method))

    target_long_only = build_long_only_top_bucket_positions(
        normalized_score,
        top_quantile=TOP_QUANTILE,
        gross_exposure=GROSS_EXPOSURE,
    )
    target_long_short = build_target_positions(
        normalized_score,
        top_quantile=TOP_QUANTILE,
        bottom_quantile=BOTTOM_QUANTILE,
        gross_exposure=GROSS_EXPOSURE,
    )

    target_by_mode = {
        "long_only_top": target_long_only,
        "long_short_top_bottom": target_long_short,
    }

    for portfolio_mode, target_positions in target_by_mode.items():
        scheduled_positions = apply_rebalance_schedule(target_positions, rebalance_frequency=REBALANCE_FREQUENCY)
        capped_positions = cap_position_weights(scheduled_positions, max_abs_weight=MAX_ABS_WEIGHT)
        if portfolio_mode == "long_only_top":
            final_positions = normalize_long_only(capped_positions, gross_exposure=GROSS_EXPOSURE)
        else:
            final_positions = renormalize_long_short(capped_positions, gross_exposure=GROSS_EXPOSURE)

        returns = compute_strategy_returns(
            positions=final_positions,
            close_prices=close_prices,
            cost_bps=COST_BPS,
            execution_lag=EXECUTION_LAG,
        )
        metrics_long = compute_portfolio_metrics(
            strategy_returns=returns,
            benchmark_returns=spy_returns,
        )
        metrics = metrics_long.set_index("metric")["value"].to_dict()
        turnover = compute_turnover(final_positions)
        active_weights = final_positions.abs().sum(axis=1).gt(0)

        portfolio_weight_records.append(panel_to_long(final_positions, "weight", portfolio_method, portfolio_mode))

        returns_output = returns.reset_index().rename(columns={"index": "Date"})
        if "Date" not in returns_output.columns:
            returns_output = returns_output.rename(columns={returns_output.columns[0]: "Date"})
        returns_output.insert(1, "portfolio_method", portfolio_method)
        returns_output.insert(2, "portfolio_mode", portfolio_mode)
        if spy_returns is not None:
            returns_output["benchmark_return"] = spy_returns.reindex(returns.index).to_numpy()
        else:
            returns_output["benchmark_return"] = np.nan
        portfolio_return_records.append(returns_output)

        portfolio_summary_records.append(
            {
                "portfolio_method": portfolio_method,
                "portfolio_mode": portfolio_mode,
                "n_survivor_alphas": len(alpha_panels),
                "n_dates": int(final_positions.shape[0]),
                "n_tickers": int(final_positions.shape[1]),
                "annualized_return": metrics.get("annualized_return"),
                "annualized_volatility": metrics.get("annualized_volatility"),
                "sharpe": metrics.get("sharpe"),
                "max_drawdown": metrics.get("max_drawdown"),
                "hit_rate": metrics.get("hit_rate"),
                "total_return": metrics.get("total_return"),
                "benchmark_total_return": metrics.get("benchmark_total_return"),
                "excess_return": metrics.get("excess_return"),
                "avg_turnover": float(turnover.mean()) if not turnover.empty else np.nan,
                "median_turnover": float(turnover.median()) if not turnover.empty else np.nan,
                "max_turnover": float(turnover.max()) if not turnover.empty else np.nan,
                "mean_gross_exposure": float(final_positions.abs().sum(axis=1).mean()),
                "mean_net_exposure": float(final_positions.sum(axis=1).mean()),
                "active_day_pct": float(active_weights.mean()) if len(active_weights) else np.nan,
                "cost_bps": COST_BPS,
                "execution_lag": EXECUTION_LAG,
                "benchmark_source": benchmark_source,
            }
        )

portfolio_alpha_scores = pd.concat(portfolio_score_records, ignore_index=True) if portfolio_score_records else pd.DataFrame(
    columns=["Date", "portfolio_method", "ticker", "combined_alpha_score"]
)
portfolio_weights = pd.concat(portfolio_weight_records, ignore_index=True) if portfolio_weight_records else pd.DataFrame(
    columns=["Date", "portfolio_method", "portfolio_mode", "ticker", "weight"]
)
portfolio_backtest_results = pd.concat(portfolio_return_records, ignore_index=True) if portfolio_return_records else pd.DataFrame(
    columns=["Date", "portfolio_method", "portfolio_mode", "strategy_return", "turnover", "cost", "benchmark_return"]
)
portfolio_performance_summary = (
    pd.DataFrame(portfolio_summary_records).sort_values(
        ["portfolio_mode", "sharpe", "total_return"],
        ascending=[True, False, False],
    ).reset_index(drop=True)
    if portfolio_summary_records
    else pd.DataFrame(
        columns=[
            "portfolio_method",
            "portfolio_mode",
            "n_survivor_alphas",
            "n_dates",
            "n_tickers",
            "annualized_return",
            "annualized_volatility",
            "sharpe",
            "max_drawdown",
            "hit_rate",
            "total_return",
            "benchmark_total_return",
            "excess_return",
            "avg_turnover",
            "median_turnover",
            "max_turnover",
            "mean_gross_exposure",
            "mean_net_exposure",
            "active_day_pct",
            "cost_bps",
            "execution_lag",
            "benchmark_source",
        ]
    )
)

print(f"Portfolio candidates: {portfolio_performance_summary.shape[0]}")
display(portfolio_performance_summary)


/var/folders/4p/d0pjwrwn08n_6l2vtvsdnzl00000gn/T/ipykernel_45442/1080097014.py:11: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  output = panel.stack(dropna=False).rename(value_name).reset_index()
/var/folders/4p/d0pjwrwn08n_6l2vtvsdnzl00000gn/T/ipykernel_45442/1080097014.py:11: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  output = panel.stack(dropna=False).rename(value_name).reset_index()
/var/folders/4p/d0pjwrwn08n_6l2vtvsdnzl00000gn/T/ipykernel_45442/1080097014.py:11: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future v

Portfolio candidates: 8


/var/folders/4p/d0pjwrwn08n_6l2vtvsdnzl00000gn/T/ipykernel_45442/1080097014.py:11: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  output = panel.stack(dropna=False).rename(value_name).reset_index()


,portfolio_method,portfolio_mode,n_survivor_alphas,n_dates,n_tickers,annualized_return,annualized_volatility,sharpe,max_drawdown,hit_rate,...,excess_return,avg_turnover,median_turnover,max_turnover,mean_gross_exposure,mean_net_exposure,active_day_pct,cost_bps,execution_lag,benchmark_source
0,equal_weight_survivors,long_only_top,1,2053,462,0.131278,0.194623,0.674525,-0.365286,0.544082,...,-0.315465,0.024172,0.0,0.240506,1.0,1.000000e+00,1.0,5,1,benchmark_prices_current.SPY
1,hybrid_survivor_weighted_portfolio,long_only_top,1,2053,462,0.131278,0.194623,0.674525,-0.365286,0.544082,...,-0.315465,0.024172,0.0,0.240506,1.0,1.000000e+00,1.0,5,1,benchmark_prices_current.SPY
2,inverse_turnover_weighted_survivors,long_only_top,1,2053,462,0.131278,0.194623,0.674525,-0.365286,0.544082,...,-0.315465,0.024172,0.0,0.240506,1.0,1.000000e+00,1.0,5,1,benchmark_prices_current.SPY
3,stress_score_weighted_survivors,long_only_top,1,2053,462,0.131278,0.194623,0.674525,-0.365286,0.544082,...,-0.315465,0.024172,0.0,0.240506,1.0,1.000000e+00,1.0,5,1,benchmark_prices_current.SPY
4,equal_weight_survivors,long_short_top_bottom,1,2053,462,0.045080,0.079885,0.564314,-0.149828,0.515343,...,-1.614870,0.019034,0.0,0.160000,1.0,6.349950e-19,1.0,5,1,benchmark_prices_current.SPY
5,hybrid_survivor_weighted_portfolio,long_short_top_bottom,1,2053,462,0.045080,0.079885,0.564314,-0.149828,0.515343,...,-1.614870,0.019034,0.0,0.160000,1.0,6.349950e-19,1.0,5,1,benchmark_prices_current.SPY
6,inverse_turnover_weighted_survivors,long_short_top_bottom,1,2053,462,0.045080,0.079885,0.564314,-0.149828,0.515343,...,-1.614870,0.019034,0.0,0.160000,1.0,6.349950e-19,1.0,5,1,benchmark_prices_current.SPY
7,stress_score_weighted_survivors,long_short_top_bottom,1,2053,462,0.045080,0.079885,0.564314,-0.149828,0.515343,...,-1.614870,0.019034,0.0,0.160000,1.0,6.349950e-19,1.0,5,1,benchmark_prices_current.SPY


## 8. Save outputs to SQLite

In [15]:
saved_paths = save_dynamic_portfolio_outputs(
    alpha_pool=portfolio_alpha_pool,
    weights=portfolio_weights,
    backtest_results=portfolio_backtest_results,
    performance_summary=portfolio_performance_summary,
    db_path=DB_PATH,
    run_id=run_id,
    portfolio_version=PORTFOLIO_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in DYNAMIC_PORTFOLIO_TABLES.items()
    ]
)

display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving portfolio outputs')


,artifact,current_table,history_table,sqlite_path
0,alpha_pool,portfolio_alpha_pool_current,portfolio_alpha_pool_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,weights,portfolio_weights_current,portfolio_weights_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,backtest_results,portfolio_backtest_results_current,portfolio_backtest_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,performance_summary,portfolio_performance_summary_current,portfolio_performance_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 9. Final summary and verification

In [16]:
decision_col = (
    "promotion_decision_final"
    if "promotion_decision_final" in portfolio_alpha_pool.columns
    else "promotion_decision"
    if "promotion_decision" in portfolio_alpha_pool.columns
    else None
)

if decision_col is None:
    portfolio_alpha_pool["promotion_decision_final"] = pd.Series(dtype="object")
    decision_col = "promotion_decision_final"

survivor_weight_columns = [
    "portfolio_method",
    "alpha_name",
    "horizon",
    "component_weight",
    "pass_rate",
    "worst_degradation",
    "avg_turnover_proxy",
    decision_col,
]
survivor_weighting_table = (
    portfolio_alpha_pool[[column for column in survivor_weight_columns if column in portfolio_alpha_pool.columns]]
    .rename(columns={decision_col: "promotion_decision"})
    .sort_values(["portfolio_method", "component_weight"], ascending=[True, False])
    if not portfolio_alpha_pool.empty
    else pd.DataFrame(columns=["portfolio_method", "alpha_name", "horizon", "component_weight", "promotion_decision"])
)

method_columns = [
    "portfolio_method",
    "portfolio_mode",
    "annualized_return",
    "sharpe",
    "max_drawdown",
    "avg_turnover",
    "total_return",
    "benchmark_total_return",
    "excess_return",
]
method_comparison = (
    portfolio_performance_summary[[column for column in method_columns if column in portfolio_performance_summary.columns]]
    .sort_values(["portfolio_mode", "sharpe"], ascending=[True, False])
    if not portfolio_performance_summary.empty
    else pd.DataFrame(columns=method_columns)
)

decision_values = sorted(portfolio_alpha_pool[decision_col].dropna().unique()) if decision_col in portfolio_alpha_pool.columns else []
method_values = set(portfolio_alpha_pool["portfolio_method"].unique()) if "portfolio_method" in portfolio_alpha_pool.columns else set()
mode_values = set(portfolio_performance_summary["portfolio_mode"].unique()) if "portfolio_mode" in portfolio_performance_summary.columns else set()
verification_checks = pd.DataFrame(
    [
        {"check": "portfolio_version", "value": PORTFOLIO_VERSION, "pass": PORTFOLIO_VERSION == "phase9_survivor_portfolio_v1"},
        {"check": "survivor_count_current", "value": len(survivor_names), "pass": len(survivor_names) == survivor_registry["alpha_name"].nunique() if "alpha_name" in survivor_registry.columns else len(survivor_names) == 0},
        {"check": "only_promote_core_survivors_used", "value": ", ".join(decision_values), "pass": set(decision_values).issubset({"PROMOTE_CORE"})},
        {"check": "review_satellite_excluded", "value": int((portfolio_alpha_pool[decision_col] == "REVIEW_SATELLITE").sum()) if decision_col in portfolio_alpha_pool.columns else 0, "pass": not ((portfolio_alpha_pool[decision_col] == "REVIEW_SATELLITE").any()) if decision_col in portfolio_alpha_pool.columns else True},
        {"check": "regime_overlays_excluded", "value": ", ".join(overlay_name_overlap), "pass": len(overlay_name_overlap) == 0},
        {"check": "portfolio_methods_created", "value": len(method_values), "pass": method_values == {"equal_weight_survivors", "stress_score_weighted_survivors", "inverse_turnover_weighted_survivors", "hybrid_survivor_weighted_portfolio"} if survivor_names else len(method_values) == 0},
        {"check": "long_only_and_long_short_supported", "value": ", ".join(sorted(mode_values)), "pass": mode_values == set(PORTFOLIO_MODES) if survivor_names else len(mode_values) == 0},
        {"check": "execution_lag_is_one_day", "value": EXECUTION_LAG, "pass": EXECUTION_LAG == 1},
        {"check": "requested_sqlite_tables_written", "value": len(DYNAMIC_PORTFOLIO_TABLES), "pass": set(DYNAMIC_PORTFOLIO_TABLES) == {"alpha_pool", "weights", "backtest_results", "performance_summary"}},
    ]
)

print("Survivor count")
display(pd.DataFrame([{"n_promote_core_survivors": len(survivor_names)}]))
print("Survivor registry used")
display(survivor_registry)
print("Survivor weighting table")
display(survivor_weighting_table)
print("Portfolio method comparison")
display(method_comparison)
print("Return / Sharpe / drawdown / turnover summary")
display(portfolio_performance_summary)
print("Benchmark comparison")
benchmark_columns = ["portfolio_method", "portfolio_mode", "benchmark_source", "benchmark_total_return", "excess_return"]
display(portfolio_performance_summary[[column for column in benchmark_columns if column in portfolio_performance_summary.columns]])
print("SQLite tables written")
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving portfolio outputs')
print("Verification checks")
display(verification_checks)


Survivor count


,n_promote_core_survivors
0,1


Survivor registry used


,survivor_id,alpha_name,horizon,alpha_sleeve,original_promotion_decision,promotion_decision_final,final_status,alpha_role,survivor_tier,survivor_selection_score,...,stress_status,source_wfv_status,failure_category,interpretation_notes,stress_version,alpha_construction_version,date_frozen,survivor_version,run_id,timestamp_frozen
0,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,CORE_REGIME,REVIEW_SATELLITE,PROMOTE_CORE,CORE_ALPHA_SURVIVOR,CORE_ALPHA,WATCH_STRESS_SURVIVOR,49.596981,...,APPROVED_STRESS,WATCHLIST_CONSTRUCTED_ALPHA_WFV,NONE,High average performance but failed catastroph...,phase7_dynamic_alpha_stress_v3,phase4a_alpha_construction_v4,2026-05-11,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437,2026-05-11 08:44:37


Survivor weighting table


...

,portfolio_method,portfolio_mode,benchmark_source,benchmark_total_return,excess_return
0,equal_weight_survivors,long_only_top,benchmark_prices_current.SPY,2.047087,-0.315465
1,hybrid_survivor_weighted_portfolio,long_only_top,benchmark_prices_current.SPY,2.047087,-0.315465
2,inverse_turnover_weighted_survivors,long_only_top,benchmark_prices_current.SPY,2.047087,-0.315465
3,stress_score_weighted_survivors,long_only_top,benchmark_prices_current.SPY,2.047087,-0.315465
4,equal_weight_survivors,long_short_top_bottom,benchmark_prices_current.SPY,2.047087,-1.614870
5,hybrid_survivor_weighted_portfolio,long_short_top_bottom,benchmark_prices_current.SPY,2.047087,-1.614870
6,inverse_turnover_weighted_survivors,long_short_top_bottom,benchmark_prices_current.SPY,2.047087,-1.614870
7,stress_score_weighted_survivors,long_short_top_bottom,benchmark_prices_current.SPY,2.047087,-1.614870


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,alpha_pool,portfolio_alpha_pool_current,portfolio_alpha_pool_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,weights,portfolio_weights_current,portfolio_weights_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,backtest_results,portfolio_backtest_results_current,portfolio_backtest_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,performance_summary,portfolio_performance_summary_current,portfolio_performance_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


Verification checks


,check,value,pass
0,portfolio_version,phase9_survivor_portfolio_v1,True
1,survivor_count_current,1,True
2,only_promote_core_survivors_used,,True
3,review_satellite_excluded,0,True
4,regime_overlays_excluded,,True
5,portfolio_methods_created,4,True
6,long_only_and_long_short_supported,"long_only_top, long_short_top_bottom",True
7,execution_lag_is_one_day,1,True
8,requested_sqlite_tables_written,4,True
